# 4. Dashboard visuals

This notebook creates **all dashboard visualization artifacts** used by the risk dashboard (BupaR, DTW, FP-Growth). Visuals are **SHAP/FFA-driven**: the original dataset is filtered to model-important features (Step 7 / Step 8) before process mining, trajectories, and itemset mining.

**Flow:** Run after [3_model_train_shap_ffa.ipynb](3_model_train_shap_ffa.ipynb). Then run [5_build_and_deploy.ipynb](5_build_and_deploy.ipynb) once to build and deploy.

## Steps

1. **Setup** – Resolve paths; create symlinks `10b_fpgrowth_dashboard_visual`, `10c_bupaR_dashboard_visual`, `10d_dtw_dashboard_visual` at repo root and under `9_risk_dashboard/visualizations` if missing.
2. **BupaR** – Process mining sequences and plots (SHAP/FFA allowed codes when available).
3. **DTW** – Trajectory features and plots (SHAP/FFA-filtered when available).
4. **FP-Growth** – Itemsets, rules, network plots (SHAP/FFA-filtered when available).
5. **Lambda/API** – Reference for dashboard endpoints.

Idempotent. Run from repo root. Prerequisites: notebook 5 done (`4_model_data`, `7_shap_analysis`, `8_ffa_analysis`); R and bupaR for BupaR.

In [ ]:
# Setup: paths and symlinks
import os
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if (REPO_ROOT / "py_helpers").exists():
    pass  # already repo root
else:
    for p in REPO_ROOT.parents:
        if (p / "py_helpers").exists():
            REPO_ROOT = p
            break
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

VISUAL_ROOT = REPO_ROOT / "9_risk_dashboard" / "visualizations"
BUPAR_SCRIPT = VISUAL_ROOT / "bupar" / "run_analysis.py"
DTW_FEATURES_SCRIPT = VISUAL_ROOT / "dtw" / "create_dtw_features.py"
DTW_ADD_SCRIPT = VISUAL_ROOT / "dtw" / "add_dtw_features_to_model_data.py"
FPGROWTH_SCRIPT = VISUAL_ROOT / "fpgrowth" / "run_analysis.py"

print(f"Repo root: {REPO_ROOT}")
print(f"Visualizations: {VISUAL_ROOT}")

In [ ]:
# Create symlinks 10b, 10c, 10d at repo root and under visualizations (idempotent: no-op if present)
def ensure_dashboard_symlinks():
    repo_links = [
        ("10c_bupaR_dashboard_visual", "9_risk_dashboard/visualizations/bupar"),
        ("10b_fpgrowth_dashboard_visual", "9_risk_dashboard/visualizations/fpgrowth"),
        ("10d_dtw_dashboard_visual", "9_risk_dashboard/visualizations/dtw"),
    ]
    for name, target in repo_links:
        path = REPO_ROOT / name
        target_path = REPO_ROOT / target
        if path.exists():
            print(f"  [repo] {name} exists")
            continue
        if not target_path.exists():
            continue
        try:
            path.symlink_to(target_path.relative_to(path.parent))
            print(f"  [repo] Created: {name}")
        except OSError as e:
            if os.name == "nt":
                print(f"  [repo] Windows: mklink /J \"{path}\" \"{target_path}\"")
            else:
                print(f"  [repo] {name}: {e}")
    for name, subdir in [("10c_bupaR_dashboard_visual", "bupar"), ("10b_fpgrowth_dashboard_visual", "fpgrowth"), ("10d_dtw_dashboard_visual", "dtw")]:
        path = VISUAL_ROOT / name
        if path.exists():
            continue
        target = VISUAL_ROOT / subdir
        if not target.exists():
            continue
        try:
            path.symlink_to(subdir)
            print(f"  [visual] Created: {name} -> {subdir}")
        except OSError as e:
            if os.name == "nt":
                print(f"  [visual] Windows: mklink /J \"{path}\" \"{target}\"")
            else:
                print(f"  [visual] {name}: {e}")

ensure_dashboard_symlinks()

## Config: cohorts and age bands

In [ ]:
from py_helpers.constants import COHORT_NAMES, AGE_BANDS

COHORTS_TO_RUN = []   # [] = all
AGE_BANDS_TO_RUN = []  # [] = all

if not COHORTS_TO_RUN:
    COHORTS_TO_RUN = COHORT_NAMES.copy()
if not AGE_BANDS_TO_RUN:
    AGE_BANDS_TO_RUN = AGE_BANDS.copy()

combinations = [(c, ab) for c in COHORTS_TO_RUN for ab in AGE_BANDS_TO_RUN]
print(f"Cohorts: {COHORTS_TO_RUN}")
print(f"Age bands: {AGE_BANDS_TO_RUN}")
print(f"Total: {len(combinations)} combinations")

## Run BupaR process mining

In [ ]:
import subprocess

FAIL_FAST = True
for cohort_name, age_band in combinations:
    print(f"\n[BupaR] {cohort_name} / {age_band}")
    result = subprocess.run(
        [sys.executable, str(BUPAR_SCRIPT), "--cohort-name", cohort_name, "--age-band", age_band],
        cwd=str(REPO_ROOT),
    )
    if result.returncode != 0 and FAIL_FAST:
        raise RuntimeError(f"BupaR failed: {cohort_name} / {age_band}")
    print(f"  -> exit {result.returncode}")

## Run DTW trajectory features

In [ ]:
for cohort_name, age_band in combinations:
    print(f"\n[DTW] {cohort_name} / {age_band}")
    r1 = subprocess.run(
        [sys.executable, str(DTW_FEATURES_SCRIPT), "--cohort", cohort_name, "--age_band", age_band],
        cwd=str(REPO_ROOT),
    )
    if r1.returncode != 0 and FAIL_FAST:
        raise RuntimeError(f"DTW create_dtw_features failed: {cohort_name} / {age_band}")
    r2 = subprocess.run(
        [sys.executable, str(DTW_ADD_SCRIPT), "--cohort-name", cohort_name, "--age-band", age_band],
        cwd=str(REPO_ROOT),
    )
    if r2.returncode != 0 and FAIL_FAST:
        raise RuntimeError(f"DTW add_dtw_features failed: {cohort_name} / {age_band}")
    print(f"  -> DTW exit {r1.returncode}, {r2.returncode}")

## Run FP-Growth

In [ ]:
for cohort_name, age_band in combinations:
    print(f"\n[FP-Growth] {cohort_name} / {age_band}")
    result = subprocess.run(
        [sys.executable, str(FPGROWTH_SCRIPT), "--cohort-name", cohort_name, "--age-band", age_band],
        cwd=str(REPO_ROOT),
    )
    if result.returncode != 0 and FAIL_FAST:
        raise RuntimeError(f"FP-Growth failed: {cohort_name} / {age_band}")
    print(f"  -> exit {result.returncode}")

## Lambda / API Gateway (reference)

Dashboard visuals are served by Lambda. Endpoints: `GET /visualizations/causal`, `/visualizations/bupar`, `/visualizations/dtw`, `/visualizations/fpgrowth`. See `9_risk_dashboard/backend/README.md` and `utility_scripts/create_api_gateway_pgx_risk_calculator.sh` to (re)create API Gateway.

In [ ]:
print("Dashboard endpoints: 9_risk_dashboard/backend/README.md")
print("API Gateway deploy: utility_scripts/create_api_gateway_pgx_risk_calculator.sh")

## Next: Build and deploy

Build and deploy run **only** in [5_build_and_deploy.ipynb](5_build_and_deploy.ipynb). Run that notebook after this one.

In [ ]:
# Build and deploy run only in 5_build_and_deploy.ipynb. Run that notebook after this one.
print("Build and deploy (once): open 5_build_and_deploy.ipynb and run it after this notebook.")

*(Build and deploy — including frontend sync to S3 — are done only in notebook 3. See above.)*

In [ ]:
# No-op: build and deploy run only in 5_build_and_deploy.ipynb
pass